## Imports
Load libraries for data processing and APIs.

In [80]:
import os                           # for environment variables (API key)
import warnings                     # suppress warnings
from pathlib import Path            # file path handling

import numpy as np                  # numerical operations
import pandas as pd                 # dataframes
import requests                     # API calls

warnings.filterwarnings("ignore")  # cleaner output
pd.set_option("display.max_columns", 100)

## Configuration
Define tickers and output paths.

In [81]:
START_DATE = "2010-01-01"
END_DATE = None

FRED_SERIES = {
    "copper_price": "PCOPPUSDM",
    "vix": "VIXCLS",
    "unrate": "UNRATE",
    "cpi": "CPIAUCSL",
    "fedfunds": "DFF",
    "indpro": "INDPRO",
    "dollar": "DTWEXBGS"
}

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

RAW_OUTPUT = OUTPUT_DIR / "copper_macro_raw_monthly.csv"
MODEL_OUTPUT = OUTPUT_DIR / "copper_macro_model_data.csv"

## Load API Key
Get FRED API key from environment (Streamlit will inject it).

In [82]:
FRED_API_KEY = os.getenv("FRED_API_KEY")
# get API key from environment (works with Streamlit)

if not FRED_API_KEY:
    raise ValueError("FRED_API_KEY not found in environment")

print("API key loaded")

API key loaded


## FRED Helper Function
Download macroeconomic data.

In [83]:
def fred_series_observations(series_id, api_key, start_date="2010-01-01"):
    url = "https://api.stlouisfed.org/fred/series/observations"

    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
        "observation_start": start_date,
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()

    data = r.json()["observations"]

    df = pd.DataFrame(data)[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df = df.set_index("date").sort_index()

    s = df["value"]
    s.name = series_id

    return s

## Download Macro Data
Fetch FRED data.

In [86]:
fred_data = {}

for name, series_id in FRED_SERIES.items():
    fred_data[name] = fred_series_observations(
        series_id,
        FRED_API_KEY,
        START_DATE
    )

macro = pd.concat(fred_data.values(), axis=1)
macro.columns = fred_data.keys()

macro shape: (5923, 6)
              vix  unrate  cpi  fedfunds  indpro  dollar
date                                                    
2026-03-16  23.51     NaN  NaN      3.64     NaN     NaN
2026-03-17  22.37     NaN  NaN      3.64     NaN     NaN
2026-03-18  25.09     NaN  NaN      3.64     NaN     NaN
2026-03-19  24.06     NaN  NaN      3.64     NaN     NaN
2026-03-20  26.78     NaN  NaN       NaN     NaN     NaN


## Create Monthly Copper Data

In [ ]:
market_monthly = pd.DataFrame()

market_monthly["copper_price"] = macro["copper_price"].resample("M").last()
# use FRED copper data and convert it to monthly frequency

## Convert Macro Data
Align to monthly frequency.

In [87]:
macro_monthly = pd.DataFrame(index=market_monthly.index)

macro_monthly["vix"] = macro["vix"].resample("M").mean()
# monthly average VIX from FRED

macro_monthly["unrate"] = macro["unrate"].resample("M").last()
macro_monthly["cpi"] = macro["cpi"].resample("M").last()
macro_monthly["indpro"] = macro["indpro"].resample("M").last()

macro_monthly["fedfunds"] = macro["fedfunds"].resample("M").mean()
macro_monthly["dollar"] = macro["dollar"].resample("M").mean()

macro_monthly shape: (195, 6)
                  vix  unrate      cpi    indpro  fedfunds      dollar
Date                                                                  
2025-11-30  19.769500     4.5  325.063  101.3605  3.876333  121.417717
2025-12-31  15.548182     4.4  326.031  101.6781  3.720645  120.188323
2026-01-31  16.179048     4.3  326.588  102.3963  3.640000  119.229810
2026-02-28  19.207000     4.4  327.460  102.5510  3.640000  117.906021
2026-03-31  24.690000     NaN      NaN       NaN  3.640000  119.413450


## Merge Data
Combine everything.

In [88]:
raw_monthly = pd.concat([market_monthly, macro_monthly], axis=1)
# merge copper and macro data

raw_monthly = raw_monthly.sort_index()
# make sure dates are in order

raw_monthly = raw_monthly.ffill()
# forward-fill small gaps in macro data

raw_monthly = raw_monthly.dropna()
# remove rows still missing after filling

raw_monthly shape: (195, 7)
            copper_price        vix  unrate      cpi    indpro  fedfunds  \
Date                                                                       
2025-11-30        5.1855  19.769500     4.5  325.063  101.3605  3.876333   
2025-12-31        5.6300  15.548182     4.4  326.031  101.6781  3.720645   
2026-01-31        5.8970  16.179048     4.3  326.588  102.3963  3.640000   
2026-02-28        6.0045  19.207000     4.4  327.460  102.5510  3.640000   
2026-03-31        5.5070  24.690000     4.4  327.460  102.5510  3.640000   

                dollar  
Date                    
2025-11-30  121.417717  
2025-12-31  120.188323  
2026-01-31  119.229810  
2026-02-28  117.906021  
2026-03-31  119.413450  


## Create Features
Generate predictive features.

In [89]:
df = raw_monthly.copy()

df["copper_ret_1m"] = df["copper_price"].pct_change()

df["copper_next_ret_1m"] = df["copper_price"].shift(-1) / df["copper_price"] - 1
df["copper_next_up"] = (df["copper_next_ret_1m"] > 0).astype(int)

df["cpi_yoy"] = df["cpi"].pct_change(12)
df["indpro_yoy"] = df["indpro"].pct_change(12)
df["dollar_yoy"] = df["dollar"].pct_change(12)

df["vix_3m_avg"] = df["vix"].rolling(3).mean()
df["vix_12m_z"] = (
    (df["vix"] - df["vix"].rolling(12).mean()) /
    df["vix"].rolling(12).std()
)

df["copper_mom_3m"] = df["copper_price"].pct_change(3)
df["copper_mom_6m"] = df["copper_price"].pct_change(6)

df.head()

df shape after features: (195, 17)
            copper_price        vix  unrate      cpi    indpro  fedfunds  \
Date                                                                       
2025-11-30        5.1855  19.769500     4.5  325.063  101.3605  3.876333   
2025-12-31        5.6300  15.548182     4.4  326.031  101.6781  3.720645   
2026-01-31        5.8970  16.179048     4.3  326.588  102.3963  3.640000   
2026-02-28        6.0045  19.207000     4.4  327.460  102.5510  3.640000   
2026-03-31        5.5070  24.690000     4.4  327.460  102.5510  3.640000   

                dollar  copper_ret_1m  copper_next_ret_1m  copper_next_up  \
Date                                                                        
2025-11-30  121.417717       0.023690            0.085720               1   
2025-12-31  120.188323       0.085720            0.047424               1   
2026-01-31  119.229810       0.047424            0.018230               1   
2026-02-28  117.906021       0.018230          

## Prepare Model Dataset
Select final columns.

In [90]:
model_data = df[[
    "copper_price",
    "copper_ret_1m",
    "copper_next_ret_1m",
    "copper_next_up",
    "vix",
    "vix_3m_avg",
    "vix_12m_z",
    "unrate",
    "cpi_yoy",
    "fedfunds",
    "indpro_yoy",
    "dollar_yoy",
    "copper_mom_3m",
    "copper_mom_6m"
]].copy()

model_data = model_data.ffill()
model_data = model_data.dropna()

## Save Data
Save outputs for modeling.

In [91]:
raw_monthly.to_csv(RAW_OUTPUT)
model_data.to_csv(MODEL_OUTPUT)

print("Data saved successfully")

Data saved successfully
